# ExSOIL walkthrough: simulation to model-data comparison

What the container can do today, end to end, with the reasoning behind each
step rather than just the commands. Written for the project team.

**The path:** run a CTSM simulation at a NEON tower site, read the output,
fetch the matching tower observations, and compare the two.

**Prerequisites.** A completed simulation somewhere the reader can find it.
If you have none, section 1 shows how to produce one; the rest of the
notebook needs it.

> **A trap worth knowing before you start.** The container ships a copy of
> `analytics_modules` at `/opt/analytics_modules`, which shadows a repository
> checkout on `sys.path`. If you are editing the code and your changes seem to
> have no effect, set `PYTHONPATH` to your checkout.

## 1. Run a simulation

`run_neon_v2` handles the whole CIME cycle -- case creation, namelist setup,
compiling CLM from Fortran, and running it:

```bash
run_neon_v2 --neon-sites KONZ --output-root ~/runs --no-batch
```

A full 2018-2024 transient at one site takes roughly six minutes of model
time on a laptop, plus about two minutes the first time to compile. The
compile is reused afterwards.

The cell below does not run a simulation. It checks whether output already
exists, because everything downstream needs some.

In [ ]:
import os
from analytics_modules import find_ctsm_hist_files

SITE = "KONZ"

# Where output is searched. Defaults to /home/user inside the container --
# the run wrappers archive there. Point it elsewhere if your run is elsewhere.
print("CTSM_OUTPUT_ROOT =", os.environ.get("CTSM_OUTPUT_ROOT", "/home/user (default)"))

try:
    files = find_ctsm_hist_files(SITE, stream="monthly")
    print(f"Found {len(files)} monthly history files for {SITE}.")
    HAVE_OUTPUT = True
except FileNotFoundError as exc:
    HAVE_OUTPUT = False
    print(f"No output for {SITE} yet.\n")
    print("Run a simulation, or point CTSM_OUTPUT_ROOT at an existing archive.")
    print("Searched:\n", exc)

## 2. Read the model output

A CTSM run does not produce one result file. It produces thousands: one per
simulated day for the daily stream, one per month for the monthly stream.
The KONZ baseline alone is over 5,000 files.

Three things vary between runs, and `open_ctsm_hist` works out all three so
you do not have to:

- **Stream naming.** CTSM 5.4 renamed the streams (`h1` became `h1a`, `h0`
  became `h0a`). Older output, including the reference copies used for
  validation, keeps the old names. Both have to stay readable, so the token
  is discovered from what is on disk rather than configured.
- **Archive layout.** Where output lands depends on which wrapper ran the
  simulation. `run_tower` archives one case flat; `run_neon_v2.py` inserts
  site and experiment segments so a perturbed run and its control stay
  separate.
- **On-disk format.** Live CTSM 5.4 output is CDF-5, which the `scipy` reader
  cannot open and `h5netcdf` cannot either. Older files are CDF-2, which
  `scipy` handles. The engine is chosen per file from its magic number.

If nothing matches, it raises and lists every path it tried, rather than
handing back an empty result that reads downstream as "no data this year".

In [ ]:
from analytics_modules import open_ctsm_hist

if HAVE_OUTPUT:
    daily = open_ctsm_hist(SITE, 2018)          # daily stream, one year
    print("dims:", dict(daily.sizes))
    print("variables:", len(daily.data_vars))

### Selecting variables

The monthly stream carries **623 variables per file**. Reading all of them
across a seven-year run takes about two minutes; asking for the one you want
takes about ten seconds, because unwanted variables are dropped as each file
opens rather than after.

Time bookkeeping is always retained regardless of what you select -- losing
the time axis to a variable filter is never what you meant.

In [ ]:
if HAVE_OUTPUT:
    monthly = open_ctsm_hist(SITE, stream="monthly", variables=["GPP"])
    print("months:", monthly.sizes["time"])
    print("GPP units:", monthly["GPP"].attrs.get("units"))

### Building a month index

**CTSM stamps monthly files at the start of the *next* month.**
`KONZ.transient.clm2.h0a.2018-07.nc` holds July's average, but its `mcdate`
is `20180801`.

So the filename, not `mcdate`, carries the month the data belongs to. Using
`mcdate` shifts the whole series forward by one month, which is easy to miss
and quietly distorts any seasonal comparison. On this data it drops the
correlation against observations from 0.87 to 0.77 and loses a month at the
boundary.

The reader handles this for you: on the monthly stream it attaches a `month`
coordinate (`YYYY-MM`) read from each file's name, so the label travels with
the values however the files were ordered. Build the index from that, never
from `mcdate`.

In [ ]:
import pandas as pd

if HAVE_OUTPUT:
    model_gpp = pd.Series(
        monthly["GPP"].squeeze().values,
        index=pd.PeriodIndex(monthly["month"].values, freq="M"),
        name="model",
    ).sort_index()

    print(model_gpp.head(3).to_string())
    print("\npeak month:", model_gpp.idxmax(), "-- should be mid-growing-season")

## 3. Fetch the tower observations

The model tells you what CTSM thinks happened. To judge it you need what the
tower measured.

Observations come from the NCAR/NEON evaluation files -- public, no
credentials, one file per site-month from 2018-01 to 2021-09.

Two things about this data are handled for you, because each is a mistake
that would otherwise be made once per notebook:

- **Units.** Observations are in µmol CO2 m⁻² s⁻¹; model GPP is in gC m⁻² s⁻¹.
  The conversion is ×12.011e-6. Getting it wrong is a five-order-of-magnitude
  error that looks like catastrophic model failure rather than a bug, so
  conversion is the default.
- **Missing data.** 18% of site-months contain no GPP at all -- and the
  quality flag does not tell you. Some files report `GPP_fqc = 0`
  ("measured") across every timestep while every value is NaN. Coverage here
  is derived from the values, never the flag, and empty months are omitted
  rather than returned as NaN.

In [ ]:
from analytics_modules import observed_gpp_coverage

coverage = observed_gpp_coverage()
print(coverage[["months_with_gpp", "months_possible", "mean_negative_fraction"]].to_string())

That table is worth pausing on. **Only KONZ has all 45 months.** ABBY has 28.
A comparison across all five sites is limited to the months every site has,
which is far fewer than it appears -- something to settle before scoping
multi-site work.

The `mean_negative_fraction` column is the other surprise: **a quarter to a
third of half-hourly observed GPP is negative**, which is physically
impossible for a gross flux.

Towers do not measure photosynthesis. They measure the *net* carbon flux, and
photosynthesis is separated out by estimating respiration and subtracting.
When that estimate runs high, the arithmetic returns a negative value. It is
estimation error, not signal.

Filtering by quality flag does not fix it (21% remain), and neither does
excluding nighttime (a third of *midday* July values are negative). Averaging
to monthly does, because the errors are symmetric noise that cancels. That is
why the comparison below is monthly, and why `monthly_observed_gpp` is the
interface rather than a half-hourly reader.

In [ ]:
from analytics_modules import monthly_observed_gpp

obs = monthly_observed_gpp(SITE)
obs.index = obs.index.to_period("M")
print(obs.head(3).to_string())
print(f"\n{len(obs)} months with data. low_signal months: {int(obs.low_signal.sum())}")

`low_signal` marks months whose mean is still below zero after averaging.
They are reported rather than clamped: all fall in the dormant season and sit
within 0.1 µmol/m²/s of zero, where model and observation are both
indistinguishable from zero. Hiding them would misrepresent the uncertainty.

## 4. Compare

Both series are now monthly and in the same units, so they can be joined
directly.

In [ ]:
if HAVE_OUTPUT:
    joined = pd.DataFrame({"model": model_gpp, "observed": obs["gpp"]}).dropna()
    print(f"overlapping months: {len(joined)}")
    print(f"correlation:        {joined.model.corr(joined.observed):.3f}")
    print(f"model / observed:   {joined.model.mean() / joined.observed.mean():.2f}")

In [ ]:
import matplotlib.pyplot as plt

if HAVE_OUTPUT:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    t = joined.index.to_timestamp()
    axes[0].plot(t, joined.model, label="model (CTSM)", lw=1.8)
    axes[0].plot(t, joined.observed, label="observed (NEON tower)", lw=1.8)
    axes[0].set_ylabel("GPP  (gC m$^{-2}$ s$^{-1}$)")
    axes[0].set_title(f"{SITE} monthly GPP")
    axes[0].legend()

    axes[1].scatter(joined.observed, joined.model, s=26)
    lim = max(joined.max()) * 1.05
    axes[1].plot([0, lim], [0, lim], "k--", lw=1, label="1:1")
    axes[1].set_xlabel("observed"); axes[1].set_ylabel("model")
    axes[1].set_title("model vs observed")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## What this does and does not tell you

**Does.** The pipeline works end to end: a simulation runs, its output is
readable without knowing which wrapper produced it, matching observations are
fetched without credentials, and the two can be compared in consistent units
at a resolution where the observation artifacts cancel.

The model tracks the seasonal cycle closely and runs high against the tower
overall. That is a plausible result and a reasonable starting point.

**Does not.** This is not a validated model evaluation.

- **No calibration.** These are baseline model settings. Hub 2's Kalman filter
  step, which adjusts parameters against observations, is not applied here.
- **No agreed tolerance.** "How close is close enough" is an open question for
  the project, so nothing here passes or fails.
- **One site.** KONZ is the only site with a complete run *and* complete
  observations. Extending to the other four needs simulations run and the
  coverage gaps above taken into account.
- **Correlation and a mean ratio are thin.** They cannot distinguish a
  seasonal-timing error from an amplitude error. Scoring the seasonal cycle
  and year-to-year variability separately is planned.

**Also worth knowing:** model GPP is *gross* photosynthesis by construction,
while tower GPP is derived from a measured net flux by partitioning. Even a
perfect model would not match exactly. Treat this as fit quality, not as
agreement.